Student Dropout Predictor

In [242]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [243]:
df = pd.read_csv("Machine_Learning_Task_Database.csv")
df.head()

,Student_ID,Attendance,Favourite_Color,Previous_Score,Random_Code,Age,Study_Hours,Gender,Lucky_Number,Absences,...,Sleep_Hours,Device,Random_Score,Academic_Warning,Extracurricular_Hours,Registration_Quarter,Random_Bucket,Assignment_Submission_Rate,Financial_Aid,Outcome
0,S149795,82.3,Green,44.0,563097.0,31.0,1.3,F,NaN,17.0,...,5.0,Tablet,58.527476,No,3.2,NaN,X4,70.1,NaN,Dropout
1,S134468,94.1,Green,NaN,620165.0,46.0,6.1,F,NaN,22.0,...,6.8,iOS,38.830845,No,3.8,1.0,X2,56.7,NO,Continue
2,S136603,64.7%,Yellow,41.0,458806.0,19.0,1.4,F,587395.0,27.0,...,7.8,Laptop,80.874824,No,13.5,1.0,X3,54.0,Yes,Dropout
3,S119995,68.9,White,32.0,250069.0,52.0,0.7,female,312645.0,8.0,...,8.6,NaN,21.141481,Yes,NaN,1.0,X2,68.8,No,Dropout
4,NaN,49.3,Red,47.0,573746.0,18.0,4.4,Female,954836.0,7.0,...,8.0,iOS,75.463445,Yes,12.4,1.0,X2,97.6,YES,Continue


In [244]:
DROP_FEATURES = [
    'Student_ID',
    'Random_Code', 'Lucky_Number', 'Random_Value', 'Random_Flag',
    'Random_Category', 'Random_Number_2', 'Random_Date_Code',
    'Random_Score', 'Random_Bucket',
    'Favourite_Color', 'Music_Preference', 'Pet_Ownership', 'Device',
    'Internet_Quality', 'City', 'Birth_Weekday', 'Registration_Quarter',
    'Age', 'Study_Hours', 'Sleep_Hours', 'Commute_Minutes',
    'Extracurricular_Hours', 'Gender', 'Income', 'Financial_Aid',
    'Education'
]
df.drop(DROP_FEATURES, axis=1, inplace=True)
df.head()

,Attendance,Previous_Score,Absences,Assignments_Completed,Stress_Level,Previous_Failures,Parent_Support,Financial_Stress,Library_Visits,Mock_Test_Score,Mentor_Meetings,Academic_Warning,Assignment_Submission_Rate,Outcome
0,82.3,44.0,17.0,1.0,10.0,3.0,4.0,7.0,8.0,77.0,5.0,No,70.1,Dropout
1,94.1,NaN,22.0,13.0,2.0,3.0,1.0,1.0,10.0,61.0,9.0,No,56.7,Continue
2,64.7%,41.0,27.0,13.0,6.0,0.0,9.0,1.0,15.0,91.0,4.0,No,54.0,Dropout
3,68.9,32.0,8.0,16.0,8.0,2.0,7.0,3.0,10.0,29.0,11.0,Yes,68.8,Dropout
4,49.3,47.0,7.0,5.0,3.0,2.0,7.0,2.0,9.0,100.0,3.0,Yes,97.6,Continue


In [245]:
FEATURES = [
    'Attendance', 'Previous_Score', 'Absences', 'Assignments_Completed',
    'Stress_Level', 'Previous_Failures', 'Parent_Support', 'Financial_Stress',
    'Library_Visits', 'Mock_Test_Score', 'Mentor_Meetings',
    'Assignment_Submission_Rate', 'Academic_Warning'
]
inputs = df[FEATURES]
le_outcome = LabelEncoder()
df["Outcome"] = le_outcome.fit_transform(df["Outcome"])
targets = df["Outcome"]

In [246]:
inputs

,Attendance,Previous_Score,Absences,Assignments_Completed,Stress_Level,Previous_Failures,Parent_Support,Financial_Stress,Library_Visits,Mock_Test_Score,Mentor_Meetings,Assignment_Submission_Rate,Academic_Warning
0,82.3,44.0,17.0,1.0,10.0,3.0,4.0,7.0,8.0,77.0,5.0,70.1,No
1,94.1,NaN,22.0,13.0,2.0,3.0,1.0,1.0,10.0,61.0,9.0,56.7,No
2,64.7%,41.0,27.0,13.0,6.0,0.0,9.0,1.0,15.0,91.0,4.0,54.0,No
3,68.9,32.0,8.0,16.0,8.0,2.0,7.0,3.0,10.0,29.0,11.0,68.8,Yes
4,49.3,47.0,7.0,5.0,3.0,2.0,7.0,2.0,9.0,100.0,3.0,97.6,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,83.0,89.0,8.0,NaN,1.0,5.0,NaN,2.0,2.0,87.0,0.0,79.3,NaN
49996,76.3,34.0,22.0,1.0,6.0,1.0,8.0,NaN,8.0,NaN,11.0,NaN,Yes
49997,50.0,91.0,16.0,13.0,4.0,1.0,9.0,10.0,10.0,97.0,9.0,53.9,Yes
49998,61.5,48.0,29.0,11.0,9.0,1.0,9.0,2.0,12.0,30.0,10.0,56.5,Yes


In [247]:
targets.head()

0    1
1    0
2    1
3    1
4    0
Name: Outcome, dtype: int64

In [248]:

cleaning_map_for_Academic_Warning = {
    'yes': 'Yes',
    'no': 'No'
}
inputs['Academic_Warning'] = inputs['Academic_Warning'].str.lower()
inputs['Academic_Warning'] = inputs['Academic_Warning'].map(cleaning_map_for_Academic_Warning)
inputs['Academic_Warning'] = inputs['Academic_Warning'].fillna(inputs['Academic_Warning'].mode()[0])
le_academic_warning = LabelEncoder()
inputs['Academic_Warning'] = le_academic_warning.fit_transform(inputs['Academic_Warning'])



In [249]:

def fill_nan_with_mean(column):
    inputs[column] = inputs[column].fillna(inputs[column].mean()).round(2)
num_list = [
    'Attendance',
    'Previous_Score',
    'Absences',
    'Assignments_Completed',
    'Stress_Level',
    'Previous_Failures',
    'Parent_Support',
    'Financial_Stress',
    'Library_Visits',
    'Mock_Test_Score',
    'Mentor_Meetings',
    'Assignment_Submission_Rate'
    ]
inputs['Attendance'] = inputs['Attendance'].astype(str).str.replace('%', '', regex=False).astype(float)
for column in num_list:
    fill_nan_with_mean(column)
inputs

,Attendance,Previous_Score,Absences,Assignments_Completed,Stress_Level,Previous_Failures,Parent_Support,Financial_Stress,Library_Visits,Mock_Test_Score,Mentor_Meetings,Assignment_Submission_Rate,Academic_Warning
0,82.30,44.00,17.0,1.00,10.0,3.0,4.00,7.00,8.0,77.00,5.0,70.10,0
1,94.10,65.12,22.0,13.00,2.0,3.0,1.00,1.00,10.0,61.00,9.0,56.70,0
2,64.70,41.00,27.0,13.00,6.0,0.0,9.00,1.00,15.0,91.00,4.0,54.00,0
3,68.90,32.00,8.0,16.00,8.0,2.0,7.00,3.00,10.0,29.00,11.0,68.80,1
4,49.30,47.00,7.0,5.00,3.0,2.0,7.00,2.00,9.0,100.00,3.0,97.60,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,83.00,89.00,8.0,9.98,1.0,5.0,5.49,2.00,2.0,87.00,0.0,79.30,1
49996,76.30,34.00,22.0,1.00,6.0,1.0,8.00,5.52,8.0,60.09,11.0,69.85,1
49997,50.00,91.00,16.0,13.00,4.0,1.0,9.00,10.00,10.0,97.00,9.0,53.90,1
49998,61.50,48.00,29.0,11.00,9.0,1.0,9.00,2.00,12.0,30.00,10.0,56.50,1


In [250]:
X_train, X_test, y_train, y_test = train_test_split(inputs, targets, test_size=0.2, random_state=42)

In [251]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)
xgb_model.fit(X_train, y_train)


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [252]:
accuracy = accuracy_score(y_test, xgb_model.predict(X_test))
accuracy

0.618

In [253]:
y_pred = xgb_model.predict(X_test)
print(classification_report(
    y_test,
    y_pred,
    target_names=["Continue", "Dropout"]
))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

    Continue       0.47      0.61      0.53      3575
     Dropout       0.74      0.62      0.68      6425

    accuracy                           0.62     10000
   macro avg       0.61      0.62      0.61     10000
weighted avg       0.65      0.62      0.63     10000

[[2190 1385]
 [2435 3990]]


In [254]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    class_weight='balanced',
    random_state=42,
)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

In [255]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(
    y_test,
    y_pred,
    target_names=["Continue", "Dropout"]
))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5983
              precision    recall  f1-score   support

    Continue       0.45      0.60      0.52      3575
     Dropout       0.73      0.60      0.66      6425

    accuracy                           0.60     10000
   macro avg       0.59      0.60      0.59     10000
weighted avg       0.63      0.60      0.61     10000

[[2157 1418]
 [2599 3826]]


In [256]:
new_student = pd.DataFrame([{
    'Attendance': 55.0,
    'Previous_Score': 42.0,
    'Absences': 22.0,
    'Assignments_Completed': 8.0,
    'Stress_Level': 8.0,
    'Previous_Failures': 3.0,
    'Parent_Support': 2.0,
    'Financial_Stress': 7.0,
    'Library_Visits': 1.0,
    'Mock_Test_Score': 38.0,
    'Mentor_Meetings': 1.0,
    'Assignment_Submission_Rate': 45.0,
    'Academic_Warning': 1   # 1 = Yes, 0 = No
}])

In [257]:
prediction = xgb_model.predict(new_student)[0]
probability = xgb_model.predict_proba(new_student)[0][1]

print("Predicted Outcome:", "Dropout" if prediction == 1 else "Continue")
print("Dropout Probability:", round(probability, 3))

Predicted Outcome: Dropout
Dropout Probability: 0.881


In [258]:
prediction = rf_model.predict(new_student)[0]
probability = rf_model.predict_proba(new_student)[0][1]

print("Predicted Outcome:", "Dropout" if prediction == 1 else "Continue")
print("Dropout Probability:", round(probability, 3))

Predicted Outcome: Dropout
Dropout Probability: 0.667


In [259]:
new_student_continue = pd.DataFrame([{
    'Attendance': 92.0,
    'Previous_Score': 85.0,
    'Absences': 3.0,
    'Assignments_Completed': 18.0,
    'Stress_Level': 2.0,
    'Previous_Failures': 0.0,
    'Parent_Support': 8.0,
    'Financial_Stress': 2.0,
    'Library_Visits': 9.0,
    'Mock_Test_Score': 88.0,
    'Mentor_Meetings': 6.0,
    'Assignment_Submission_Rate': 95.0,
    'Academic_Warning': 0
}])

prediction = xgb_model.predict(new_student_continue)[0]
probability = xgb_model.predict_proba(new_student_continue)[0][1]

print("Predicted Outcome:", "Dropout" if prediction == 1 else "Continue")
print("Dropout Probability:", round(probability, 3))

Predicted Outcome: Continue
Dropout Probability: 0.122
